# Figure design
Open this notebook from codes/day-2. Synthetic households; EUR per month.


## The same 195 households
Open 05_design.ipynb. Money amounts are EUR per month.


In [ ]:
from pathlib import Path
import pandas as pd
import plotnine as pn
from lesson_data import load_households
df = load_households()
print("Households:", len(df))
print("Negative saving:", (df["saving"] < 0).sum())


## The default scatterplot
Which units belong on each axis? Which dots overlap?


In [ ]:
base = pn.ggplot(df,
    pn.aes("income", "saving"))
default = base + pn.geom_point()
default


## Axis labels and a useful title
The variable names become labels the reader can interpret.


In [ ]:
labelled = base + pn.labs(
    title="Household saving and income",
    x="Monthly income (EUR)",
    y="Monthly saving (EUR)",
)
p = labelled + pn.geom_point()
p


## Point size and transparency
alpha = 0.55 makes overlapping observations darker.


In [ ]:
points = labelled + pn.geom_point(
    color="#286482",
    alpha=0.55,
    size=1.8,
)
points


## The zero-saving reference
Dots below the line have consumption greater than income.


In [ ]:
zero = points + pn.geom_hline(
    yintercept=0,
    color="#777777",
    size=0.5,
)
zero


## A quieter background
Minor gridlines disappear. Major gridlines still help compare amounts.


In [ ]:
clean = (
    zero
    + pn.theme_minimal(base_size=15)
    + pn.theme(
        panel_grid_minor=(
            pn.element_blank())
    )
)
clean


## The caption identifies the sample
195 observations have usable income and consumption.


In [ ]:
finished = clean + pn.labs(
    caption=(
        "Synthetic classroom data"
        f" | n = {len(df)} households"
    )
)
finished


## Exercise 1 · Consumption and income
Build a scatterplot of consumption against income. Use pn.geom_abline(slope=1, intercept=0). What does a dot above the line mean?


In [ ]:
# Use the same df.
# Put income on x and consumption on y.
# Add a 45-degree reference line.
# Use labels, alpha, a theme, and a caption.


## Worked answer · Consumption and income
Above the diagonal, consumption exceeds income.


In [ ]:
spending = (
    pn.ggplot(df,
        pn.aes("income", "consumption"))
    + pn.geom_abline(slope=1, intercept=0)
    + pn.geom_point(alpha=0.5,
        color="#286482")
    + pn.labs(x="Monthly income (EUR)",
        y="Monthly consumption (EUR)",
        caption=("Synthetic households"
            " | n=195"))
    + pn.theme_minimal(base_size=15)
)
spending


## Colour identifies a variable
The region column creates the groups and legend.


In [ ]:
by_region = (
    pn.ggplot(df,
        pn.aes("income", "saving",
               color="region"))
    + pn.geom_point(alpha=0.65)
    + pn.labs(x="Income (EUR)",
        y="Saving (EUR)", color="Region")
    + pn.theme_minimal(base_size=15)
)
by_region


## An explicit four-colour palette
Use region names to keep colours consistent across figures.


In [ ]:
palette = {
    "East": "#0072B2",
    "North": "#D55E00",
    "South": "#009E73",
    "West": "#CC79A7",
}
coloured = (
    by_region
    + pn.scale_color_manual(
        values=palette)
)
coloured


## Common axes across panels
Each panel uses the same income and saving scales.


In [ ]:
panels = (
    coloured
    + pn.facet_wrap("region", ncol=2)
    + pn.theme(legend_position="none")
)
panels


## The numbers behind a comparison
A bar height will represent mean monthly saving.


In [ ]:
summary = df.groupby("region", as_index=False).agg(
    households=("household_id", "size"),
    mean_saving=("saving", "mean"),
)
print(summary.round(1).to_string(index=False))


## Alphabetical category order
Alphabetical order gives no immediate ranking by saving.


In [ ]:
bars = (
    pn.ggplot(summary,
        pn.aes("region", "mean_saving"))
    + pn.geom_col(fill="#286482")
    + pn.labs(x="Region",
        y="Mean monthly saving (EUR)")
    + pn.theme_minimal(base_size=15)
)
bars


## Ordering is part of the data
The categorical order controls the discrete axis.


In [ ]:
order = summary.sort_values(
    "mean_saving"
)["region"].tolist()
ordered = summary.copy()
ordered["region"] = pd.Categorical(
    ordered["region"], categories=order,
    ordered=True,
)
ordered["value"] = ordered["mean_saving"].round(1)
print(order)


## Horizontal bars and direct labels
Read exact values without estimating the end of each bar.


In [ ]:
ranked = (
    pn.ggplot(ordered,
        pn.aes("region", "mean_saving"))
    + pn.geom_col(fill="#286482")
    + pn.geom_text(pn.aes(label="value"),
        ha="left", nudge_y=8, size=11)
    + pn.expand_limits(y=430)
    + pn.coord_flip()
    + pn.labs(x="",
        y="Mean saving (EUR/month)")
    + pn.theme_minimal(base_size=15)
)
ranked


## A zoom excludes some values from view
The selected subset has a different median.


In [ ]:
inside = df["saving"].between(0, 1000)
print("All households:", len(df))
print("Within 0 to 1000:", inside.sum())
print("Median, all:", df["saving"].median())
print("Median, selected:", df.loc[
    inside, "saving"].median())


## Scale limits change boxplot statistics
Scale limits remove observations before the boxplot calculates quartiles.


In [ ]:
box = (
    pn.ggplot(df,
        pn.aes("region", "saving"))
    + pn.geom_boxplot(fill="#A9CBDC")
    + pn.theme_minimal(base_size=15)
)
limited = box + pn.scale_y_continuous(
    limits=(0, 1000)
)
limited


## Coordinate limits zoom the full boxplot
Each region’s quartiles use its full sample. Only the display window changes.


In [ ]:
zoomed = box + pn.coord_cartesian(
    ylim=(0, 1000)
)
zoomed


## Exercise 2 · Rank median income
Create a ranked comparison of median monthly income per person. Explain why it answers a different question from mean household saving.


In [ ]:
# Calculate median income per person by region.
# Order the categories by that median.
# Make a horizontal bar or point chart.
# Label the units and display each value.


## Worked answer · The median table
Each region contributes one median income per person.


In [ ]:
medians = df.groupby("region", as_index=False).agg(
    median_income=("income_per_person", "median")
)
order = medians.sort_values(
    "median_income")["region"].tolist()
medians["region"] = pd.Categorical(
    medians["region"], categories=order,
    ordered=True,
)
medians["value"] = medians["median_income"].round(1)
print(medians.round(1).to_string(index=False))


## Worked answer · The ranked plot
Income per person and saving measure different aspects of a household budget.


In [ ]:
median_plot = (
    pn.ggplot(medians,
        pn.aes("region", "median_income"))
    + pn.geom_col(fill="#0072B2")
    + pn.geom_text(pn.aes(label="value"),
        ha="left", nudge_y=15, size=11)
    + pn.expand_limits(y=1200)
    + pn.coord_flip()
    + pn.labs(x="", y=("Median income"
        " (EUR/person/month)"))
    + pn.theme_minimal(base_size=15)
)
median_plot


## PNG for slides, PDF for print
The plot code stays editable. The exported files record the chosen size.


In [ ]:
output = Path("outputs")
output.mkdir(exist_ok=True)
median_plot.save(
    output / "median-income.png",
    width=9, height=5, dpi=200, verbose=False,
)
median_plot.save(
    output / "median-income.pdf",
    width=9, height=5, verbose=False,
)
print((output / "median-income.png").is_file())
